In [ ]:
from pathlib import Path
import os
import sys

# Resolve paths once so every notebook uses the project folder.
project_folder = next((
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / 'project_code_2026_publicshare',
                   parent / 'project_2026/project_code_2026_publicshare',
                   parent / 'private_github/project_2026/project_code_2026_publicshare')
    if (folder / 'pricing_helpers.py').is_file()
), None)
if project_folder is None:
    raise FileNotFoundError('Open the project folder as your notebook workspace.')
os.chdir(project_folder)
sys.path.insert(0, str(project_folder))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In this document, we help you understand the training data for the entire project, for you to use for your demand estimation and pricing model. We also describe the test data (for which you have to give optimal prices).

The training data is composed of 50000 customers, inside `train_prices_decisions_2026.csv`.

For every training customer, you have:
1. a price they were offered for the item.
2. their decision for whether they purchased: `False` means they did not purchase the item, `True` means they purchased the item. 
3. three user covariates for each user to aid in personalized pricing. 
`Covariate1` (feature 0) is binary.


In [ ]:
train_pricing_decisions = pd.read_csv(project_folder / 'data/train_prices_decisions_2026.csv')
test_user_info = pd.read_csv(project_folder / 'data/test_user_info_2026.csv')

In [ ]:
train_pricing_decisions.head()

In [ ]:
train_pricing_decisions.price_item.hist(bins = 100)
plt.yscale('log')

The following plot shows the average demand for each price offered:

In [ ]:
train_pricing_decisions.groupby(
    pd.qcut(train_pricing_decisions.price_item, 20))['item_bought'].mean().plot(kind='line')
_ = plt.xticks(rotation=45)

Of course, you might consider doing something more clever in estimating the demand, using the provided covariates. For example, customers have very different demand depending on their second covariate:

In [ ]:
high_mask = train_pricing_decisions.Covariate2 > train_pricing_decisions.Covariate2.mean()
train_pricing_decisions.loc[high_mask].groupby(
    pd.qcut(train_pricing_decisions.loc[high_mask].price_item, 20))['item_bought'].mean().plot(kind='line', label = 'High Covariate2')
train_pricing_decisions.loc[~high_mask].groupby(
    pd.qcut(train_pricing_decisions.loc[~high_mask].price_item, 20))['item_bought'].mean().plot(kind='line', label = 'Low Covariate2')
_ = plt.legend()
_ = plt.xticks(rotation=45)

The test data just has user covariates. Your job is to calculate a price for each customer so as to maximize expected revenue from that customer, as you did in Homework 3.

In [ ]:
test_user_info.head()

For Part 1, submit one CSV directly in `agents/`, named `static_prices_submission.csv`, with three required columns: `user_index`, `price_item`, and `expected_revenue`. Include every test customer exactly once; all required entries must be finite, nonnegative numbers.
